In [1]:
import re
import pandas as pd
from fuzzywuzzy import fuzz

/home/linux/miniconda3/envs/anpr_prod_cpu/lib/python3.8/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


- Assuming Maharashtrian number plate eg. MH11CW7007

- Length should be 9 or 10 characters
- First two characters show be alphabet and it should be belonging to the state list and if not then it should be updated accordingly
- second and third characters should be numberical and count varies depending on state
- Next One or Two characters should be alphabet
- Next four characters should be numberical


In [2]:
str = 'MH11CV4142'
found = re.findall(r"^[A-Z]{2}[0-9]{2}[A-Z]{1,2}[0-9]{1,4}$",str)
print(len(found)>0)

True


In [3]:
# raw_data.shape

In [13]:
# raw_data=pd.read_excel('../data/csvs/ANPR_Analysis_2.xlsx',engine='openpyxl')
raw_data=pd.read_excel('output_1384.xlsx',engine='openpyxl')

# raw_data=pd.read_csv('../data/csvs/Analysis_2.csv')

raw_data.head()

,id,datetime,vehicleno,processed,match,sync,clat,clong,location,vehTypeId,userId,locationId,rfid
0,1,08:07:2023_13:29:56,MH12HD1951,NaN,NaN,1,23.3636,72.3336,Supa Toll,0,1,0,34161FA820328EE815B3EC00
1,2,08:07:2023_13:30:19,MH12SX0922,NaN,NaN,1,23.3636,72.3336,Supa Toll,0,1,0,34161FA820328EE815B3EC00
2,3,NaN,NaN,NaN,NaN,1,23.3636,72.3336,Supa Toll,0,1,0,34161FA820328EE815B3EC00
3,4,08:07:2023_13:33:35,MH14KA5869,NaN,NaN,1,23.3636,72.3336,Supa Toll,0,1,0,34161FA820328EE815B3EC00
4,5,08:07:2023_13:34:28,MH16CC4728,NaN,NaN,1,23.3636,72.3336,Supa Toll,0,1,0,34161FA820328EE815B3EC00


In [5]:
raw_data.shape

(1383, 11)

In [6]:
state_code_dict={
    'AN':['AN','AM'],
    'AP':['AP'],
    'AR':['AR','AB'],
    'AS':['AS','A5'],
    'BR':['BR','BB','88','8R'],
    'CG':['CG','C6','GG'],
    'CH':['CH','CN','CM'],
    'DD':['DD','D0','OO','00'],
    'DL':['DL','0L','OL'],
    'GA':['GA','6A'],
    'GJ':['GJ','6J','GI'],
    'HP':['HP'],
    'HR':['HR','MR','NR','HH','HK'],
    'JH':['JH','JM','JN'],
    'JK':['JK'],
    'KA':['KA'],
    'KL':['KL'],
    'LA':['LA'],
    'LD':['LD','L0','LO'],
    'MH':['MH','NH','WH','HW','AH','KH','MA','M8','MF','M4','MB'],
    'ML':['ML','HL'],
    'MN':['MN','MM'],
    'MP':['MP','N1'],
    'MZ':['MZ','NZ','M2','N2'],
    'NL':['NL','NC'],
    'OD':['OD','0D','O0'],
    'PB':['PB','P8'],
    'PY':['PY'],
    'RJ':['RJ','RU','8J'],
    'SK':['SK','5K'],
    'TN':['TN','TH','TM'],
    'TR':['TR'],
    'TS':['TS','T5'],
    'UK':['UK','YK'],
    'UP':['UP'],
    'WB':['WB','W8']
 }

In [7]:
dist_code_dict={
    '0':['0','O','Q','D'],
    '1':['1','I'],
    '2':['2','Z'],
    '3':['3'],
    '4':['4'],
    '5':['5','S'],
    '6':['6','G','E'],
    '7':['7','J','T'],
    '8':['8','B'],
    '9':['9'],
}

In [8]:
in_series_dict={
    'A':['A'],
    'B':['B','8'],
    'C':['C'],
    'D':['D'],
    'E':['E'],
    'F':['F'],
    'G':['G','6'],
    'H':['H'],
    'I':['I','1'],
    'J':['J'],
    'K':['K'],
    'L':['L'],
    'M':['M'],
    'N':['N'],
    'O':['O','0'],
    'P':['P'],
    'Q':['Q'],
    'R':['R'],
    'S':['S','5'],
    'T':['T'],
    'U':['U'],
    'V':['V'],
    'W':['W'],
    'X':['X'],
    'Y':['Y'],
    'Z':['Z','2']
}

In [9]:
str_nh='0001C9656'
state_code=''
dist_code=''
series=''
last_number=''
if len(str_nh)>=9:
    state_code=str_nh[:2]
    dist_code=str_nh[2:4]
    series=str_nh[4:6]
    last_number=str_nh[6:]
    
print('state_code : ',state_code)
print('dist_code : ',dist_code)
print('series : ',series)
print('last_number : ',last_number)



state_code :  00
dist_code :  01
series :  C9
last_number :  656


In [10]:
def process_state_code(in_state_code):
    if in_state_code in state_code_dict.keys():
        return in_state_code
    else:
        for state_code_txt,data_lst in state_code_dict.items():
            if in_state_code in data_lst:
                return state_code_txt
        return 'Manual Check Required'

def process_dist_code(in_dist_code):
    final_dist_code=''
    for number in in_dist_code:
        if number in dist_code_dict.keys():
            final_dist_code+=number
        else:
            for dist_code_txt,data_lst in dist_code_dict.items():
                if number in data_lst:
                    final_dist_code+=dist_code_txt
    # print('final_dist_code : ',final_dist_code)
    if (len(final_dist_code)==2 or len(final_dist_code)==4) and final_dist_code.isnumeric():
        return final_dist_code
    else:
        return 'Manual Check Required'
    
def process_series(in_series):
    final_series_code=''
    for char_txt in in_series:
        if char_txt in in_series_dict.keys():
            final_series_code+=char_txt
        else:
            for series_code_txt,data_lst in in_series_dict.items():
                if char_txt in data_lst:
                    final_series_code+=series_code_txt
    if (len(final_series_code)==2 or len(final_series_code)==1) and final_series_code.isalpha():
        return final_series_code
    else:
        return 'Manual Check Required'

In [28]:
class PostProcessing():
    def __init__(self) -> None:
        pass
    def process_state_code(self,in_state_code):
        print('in_state_code : ',in_state_code)
        if in_state_code in state_code_dict.keys():
            return in_state_code,0
        else:
            for state_code_txt,data_lst in state_code_dict.items():
                if in_state_code in data_lst:
                    return state_code_txt,0
            return in_state_code,1#'Manual Check Required'

    def process_dist_code(self,in_dist_code):
        final_dist_code=''
        for number in in_dist_code:
            if number in dist_code_dict.keys():
                final_dist_code+=number
            else:
                for dist_code_txt,data_lst in dist_code_dict.items():
                    if number in data_lst:
                        final_dist_code+=dist_code_txt
        # print('final_dist_code : ',final_dist_code)
        if len(final_dist_code)>=2 and final_dist_code.isnumeric():
            return final_dist_code,0
        else:
            return final_dist_code,1#'Manual Check Required'
        
    def process_series(self,in_series):# one or two chars
        final_series_code=''
        for index,char_txt in enumerate(in_series):
            if char_txt.isnumeric() and index==1:
                final_series_code+=char_txt
            elif char_txt in in_series_dict.keys():
                final_series_code+=char_txt
            else:
                for series_code_txt,data_lst in in_series_dict.items():
                    if char_txt in data_lst:
                        final_series_code+=series_code_txt
        if len(final_series_code)==2:
            return final_series_code,0
        else:
            return final_series_code,1#'Manual Check Required'
    def Replace_IO_dist_code(self):
        self.dist_code=self.dist_code.replace('I','1')
        self.dist_code=self.dist_code.replace('O','0')
        

    def main(self,Predicted_number_plate):
        Further_analysis_required_Flag=0
        # print('Predicted_number_plate : ',Predicted_number_plate)
        processed_str=''
        if len(Predicted_number_plate)>8 and len(Predicted_number_plate)<11:#9 and 10
            self.state_code=Predicted_number_plate[:2]
            self.dist_code=Predicted_number_plate[2:4]
            self.series=Predicted_number_plate[4:6]
            self.last_number=Predicted_number_plate[6:]

            if self.state_code.isnumeric() and self.dist_code=='BH':
                return Predicted_number_plate
            
            self.Replace_IO_dist_code()
            process_state_code_output,process_state_code_Flag=self.process_state_code(self.state_code)
            print('process_state_code_output : ',process_state_code_output)
            processed_str+=process_state_code_output

            process_dist_code_output,process_dist_code_Flag=self.process_dist_code(self.dist_code)
            processed_str+=process_dist_code_output
            
            process_series_output,process_series_Flag=self.process_series(self.series)
            processed_str+=process_series_output
            
            process_dist_code_output_last_series,process_dist_code_last_series_Flag=self.process_dist_code(self.last_number)
            processed_str+=process_dist_code_output_last_series

            if (process_state_code_Flag or process_dist_code_Flag or process_series_Flag or process_dist_code_last_series_Flag)==1:
                 return processed_str,'Further analysis required'
            else:
                return processed_str,'Success'
            
            # processed_str+=self.process_state_code(self.state_code)
            # processed_str+=self.process_dist_code(self.dist_code)
            # processed_str+=self.process_series(self.series)
            # processed_str+=self.process_dist_code(self.last_number)
            # if 'Manual Check Required' in processed_str:
            #     return processed_str,'Further analysis required'
            # else:
            #     return processed_str,'Success'

        elif len(Predicted_number_plate)>10:
            for i in range(len(Predicted_number_plate)-1):
                check_state_str=Predicted_number_plate[i:i+2] 
                if 'Manual Check Required' in self.process_state_code(check_state_str):
                    pass
                else:
                    Predicted_number_plate=Predicted_number_plate[i:]
                    if len(Predicted_number_plate)>8 and len(Predicted_number_plate)<11:#9 and 10
                        self.state_code=Predicted_number_plate[:2]
                        self.dist_code=Predicted_number_plate[2:4]
                        self.series=Predicted_number_plate[4:6]
                        self.last_number=Predicted_number_plate[6:]
                        processed_str+=self.process_state_code(self.state_code)[0]
                        processed_str+=self.process_dist_code(self.dist_code)[0]
                        processed_str+=self.process_series(self.series)[0]
                        processed_str+=self.process_dist_code(self.last_number)[0]
                        if 'Manual Check Required' in processed_str:
                            return processed_str,'Further analysis required'
                        else:
                            return processed_str,'Success'
                    else:
                        return Predicted_number_plate,'Length < 9 clip Error'



            
        else:
            return Predicted_number_plate,'>8 and < 11 Error'
            

In [29]:
PP_obj=PostProcessing()
final_data_lst=[]
for index,data in raw_data.iterrows():
    temp=data
    temp['comment']=''
    try:
        # print(data)
        if len(data['vehicleno'])>0:
            processed_str,PostProcessed_Flag=PP_obj.main(data['vehicleno'].upper())
            print('processed_str : ',processed_str)
            temp['processed']=processed_str
            temp['comment']=PostProcessed_Flag
            data['match']=fuzz.ratio(processed_str.upper(),data['vehicleno'].upper())

        final_data_lst.append(temp)
    except Exception as e:
        # print(e)
        # raise
        continue
    # break
# print(final_data_lst)
pd.DataFrame(final_data_lst).to_excel('Output_Post_processed_1384_1.xlsx',index=False)


in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH12HD1951
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH12SX0922
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH14KA5869
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH16CC4728
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH16CC7359
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH16CD5637
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH46BB8656
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH45S1004
in_state_code :  RJ
process_state_code_output :  RJ
processed_str :  RJ32GC6410
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH06CD1811
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH04FD8230
in_state_code :  MH
process_state_code_output :  MH
processed_str :  MH15CD1919
in_state_code :  MH
process_state_code_ou

In [12]:
import pandas as pd

In [18]:
df=pd.read_excel('../data/csvs/ANPR_Analysis_2.xlsx')
df.head()

,Image_Path,Image_Name,Ground_Truth,Predicted,Raw_Prediction,Match%,Match_Individual%,Detection,craft,Text,NP_crop_paths,craft_crop_paths,ANPR_Final_Output
0,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,0PB08ER8794-anpr-raw-lane2_34.jpg,PB08ER8794,P808ER8794,['P808ER8794'],90.0,[90],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/0PB08ER8794-anpr-raw-lane...,../data/NP_Recognition/0PB08ER8794-anpr-raw-la...
1,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,AP07TB3589-anpr-raw-lane2_30.jpg,AP07TB3589,APO7TB3589,['APO7TB3589'],90.0,[90],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/AP07TB3589-anpr-raw-lane2...,../data/NP_Recognition/AP07TB3589-anpr-raw-lan...
2,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,AP24TB3436-anpr-raw-lane4_80.jpg,AP24TB3436,AP24FTB3436,['AP24FTB3436'],95.0,[95],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/AP24TB3436-anpr-raw-lane4...,../data/NP_Recognition/AP24TB3436-anpr-raw-lan...
3,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,AP29TB4995-anpr-raw-lane2_13.jpg,AP29TB4995,AP29TB4995,['AP29TB4995'],100.0,[100],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/AP29TB4995-anpr-raw-lane2...,../data/NP_Recognition/AP29TB4995-anpr-raw-lan...
4,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,BR01GK2670-anpr-raw-lane2_93.jpg,BR01GK2670,BRO1GK2670,['BRO1GK2670'],90.0,[90],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/BR01GK2670-anpr-raw-lane2...,../data/NP_Recognition/BR01GK2670-anpr-raw-lan...


In [19]:
df1=pd.read_excel('../data/csvs/To_Analyze_Name_3877.xlsx')
df1.head()

/home/linux/miniconda3/envs/anpr_prod_cpu/lib/python3.8/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Image_Name,Corrected_Image_Name,Comment,Unnamed: 3
0,0PB08ER8794-anpr-raw-lane2_34.jpg,PB08ER8794-anpr-raw-lane2_34.jpg,NaN,NaN
1,AP24TB3436-anpr-raw-lane4_80.jpg,NaN,NaN,NaN
2,CG04JD3395-anpr-raw-lane4_25.jpg,NaN,NaN,NaN
3,CG04LG5651-anpr-raw-lane4_21.jpg,NaN,NaN,NaN
4,CG04NT5120-anpr-raw-lane4_37.jpg,NaN,NaN,NaN


In [27]:
# df1.Image_Name.to_list()
new_gt=[]
for item in df1.Corrected_Image_Name.str.split('-'):
    try:
        new_gt.append(item[0])
    except:
        new_gt.append('')



In [30]:
len(new_gt)
len(df1.Image_Name.to_list())

3877

In [33]:
update_data=dict(zip(df1.Image_Name.to_list(),new_gt))

In [34]:
df.columns

Index(['Image_Path', 'Image_Name', 'Ground_Truth', 'Predicted',
       'Raw_Prediction', 'Match%', 'Match_Individual%', 'Detection', 'craft',
       'Text', 'NP_crop_paths', 'craft_crop_paths', 'ANPR_Final_Output'],
      dtype='object')

In [62]:

final_df=[]
for index,data in df.iterrows():

    Saved_Image_Name=data['Image_Name']
    # print('Saved_Image_Name : ',Saved_Image_Name)
    # if Saved_Image_Name!='MH16CD1895-anpr-raw-lane4_58.jpg':
    #     continue
    if Saved_Image_Name in update_data.keys() and len(update_data[Saved_Image_Name])>0:
        data['Ground_Truth_1']=update_data[Saved_Image_Name]
        data['Predicted_1']=data['Predicted']
        try: 
            data['Match%_1']=fuzz.ratio(data['Predicted_1'],data['Ground_Truth_1'])
        except:
            data['Match%_1']='Error'
        # print(data)
    else:
        data['Ground_Truth_1']=Saved_Image_Name.split('-')[0]
        data['Predicted_1']=data['Predicted']
        try: 
            data['Match%_1']=fuzz.ratio(data['Predicted_1'],data['Ground_Truth_1'])
        except:
            data['Match%_1']='Error'
        # print(data)
    final_df.append(data)
    

In [63]:
df_final=pd.DataFrame(final_df)

In [64]:
df_final[df_final['Image_Name']=='MH16CD1895-anpr-raw-lane4_58.jpg']

,Image_Path,Image_Name,Ground_Truth,Predicted,Raw_Prediction,Match%,Match_Individual%,Detection,craft,Text,NP_crop_paths,craft_crop_paths,ANPR_Final_Output,Ground_Truth_1,Predicted_1,Match%_1
850,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane4_58.jpg,MH16CD1404,NH16CD1895,['NH16CD1895'],60.0,[60],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane4...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,NH16CD1895,90


In [65]:
#MH16CD1895
df_final[df_final['Ground_Truth']=='MH16CD1404']

,Image_Path,Image_Name,Ground_Truth,Predicted,Raw_Prediction,Match%,Match_Individual%,Detection,craft,Text,NP_crop_paths,craft_crop_paths,ANPR_Final_Output,Ground_Truth_1,Predicted_1,Match%_1
850,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane4_58.jpg,MH16CD1404,NH16CD1895,['NH16CD1895'],60.0,[60],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane4...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,NH16CD1895,90
2349,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane2_33.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane2...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
2391,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16DC1741-anpr-raw-lane2_41.jpg,MH16CD1404,MH16CD1404,['MH16CD1404'],100.0,[100],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16DC1741-anpr-raw-lane2...,../data/NP_Recognition/MH16DC1741-anpr-raw-lan...,MH16CD1404,MH16CD1404,100
3621,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane2_24.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane2...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
4628,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane2_36.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane2...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
6185,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane2_72.jpg,MH16CD1404,MH16CD1404,['MH16CD1404'],100.0,[100],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane2...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1404,MH16CD1404,100
6186,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane2_73.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane2...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
6908,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane3_72.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane3...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
7999,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane4_61.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane4...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100
12458,/home/linux/DeepLearning/Uday/Toll/Toll_Data_O...,MH16CD1895-anpr-raw-lane3_77.jpg,MH16CD1404,MH16CD1895,['MH16CD1895'],70.0,[70],Done,Done,Done,../data//NP_detection_pred//crops/Number_Plate...,../data/crops_folder/MH16CD1895-anpr-raw-lane3...,../data/NP_Recognition/MH16CD1895-anpr-raw-lan...,MH16CD1895,MH16CD1895,100


In [67]:
df_final.to_excel('../data/csvs/ANPR_Analysis_2_Updated.xlsx',index=False)
